# Getting Started with *mibitrans*

Simple hydrogeological contaminant transport modelling with the *mibitrans* package. For a detailed outline of the *mibitrans* functionality, including explanation see the notebook `basic_use_of_mibitrans.ipynb`.

**Authors:** Alraune Zech, Jorrit Bakker

In [ ]:
import matplotlib.pyplot as plt
import mibitrans as mbt

## Run your model within a few lines of code!

The following cell shows all the python code you need to run the complete transport simulation.

In [ ]:
hydro = mbt.HydrologicalParameters(velocity = 0.1,  # [m/d]
                                   porosity = 0.3,  # [-]
                                   alpha_x = 3,     # [m]
                                   alpha_y = 0.02,  # [m]
                                   alpha_z = 0.005) # [m]
att = mbt.AttenuationParameters(retardation = 1.3,  # [-]
                                half_life = 182.5)  # [d]
source = mbt.SourceParameters(depth = 2,            # [m] 
          source_zone_boundary = [5, 10, 20],       # [m]  
          source_zone_concentration = [10, 4, 0.5]) # [g/m^3]
model = mbt.ModelParameters(model_length = 100,     # [m]
                            model_width = 60,       # [m]
                            model_time = 5 * 365)   # [d]
mbt_object = mbt.Mibitrans(hydro,att,source,model)
mbt_results = mbt_object.run()
mbt_results.centerline(time = 0.2*365, label ='t = 0.2y')
mbt_results.centerline(time = 1*365, label ='t = 1y')
mbt_results.centerline(label = 't = 5y')
plt.legend()
plt.show()

## What did the code do?

You have defined your parameters in the four input dataclass, linked to the type of process: hydrological parameters (`hydro`), attenuation parameters (`att`), source parameters (`source`), model parameters (`model`). Units of parameters have default values, consistent between data input categories.

You have initialized the model object (`mbt_object`) chosing the `Mibitrans` model class. Then you ran the model with `mbt_object.run()` generating an output object `mbt_results` that contains the concentration distribution as 3D numpy array (indexed as $C[t,y,x]$) and the input and calculated parameters used for this specific run. 

Finally, you have plotted the concentration distribution along the plume center line for various times.

## Additional Options

Use a different transport model class (`Anatrans`) for the same set of parameters and display the concentration as 2D plume.

In [ ]:
ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results = ana_object.run()
ana_results.plume_2d(cmap="Spectral", shading="gouraud")

Use another transport model class (`Bioscreen`) for the same set of parameters and display the concentration as 3D plume.

In [ ]:
bio_object = mbt.Bioscreen(hydro,att,source,model)
bio_results = bio_object.run()
ax = bio_results.plume_3d(cmap="cividis")

Calculate concentrations under different degredation options (for you `Mibitrans` transport model) and compare results as breakthrough curve.

In [ ]:
mbt_object.attenuation_parameters.decay_rate = 0
mbt_results_nd = mbt_object.run()

In [ ]:
mbt_object_inst = mbt.Mibitrans(hydro,att,
                                source,model)
mbt_object_inst.instant_reaction(
    electron_acceptors={"delta_oxygen":9,
    "delta_nitrate":6,"ferrous_iron":8,
    "delta_sulfate":5,"methane":4})
mbt_results_inst = mbt_object_inst.run()

In [ ]:
mbt.breakthrough([mbt_results,mbt_results_nd,mbt_results_inst],
                 x_position=50,lw = 2, marker = 'o',mec = 'k')
plt.legend(['lin decay','no decay','inst. reaction'])